# Propeller noise

The motor harmonics of the drone between 37 and 60 Hz, tracked with the synchrosqueezed STFT.
Surveys 7 to 11 are flown at 1, 1.5, 2, 2.5 and 3 m/s to relate the ridge frequencies to the velocity.
All processing lives in `src/propellers.py`; this notebook only plots.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import propellers as pr
from dataloader import load_survey
from signal_processing_tools import FS, normal_fit

## One flight

In [ ]:
SURVEY_INDEX = 7
SENSOR = 3

survey = load_survey(SURVEY_INDEX, pr.SURVEY_SPEED)
data = pr.track_data(survey, SENSOR, trim=(3000, -5000), exclude_tracks=[])
print("Tracks:", sorted(data["Track_number"].unique()))

plt.scatter(data["Longitude [m]"], data["Latitude [m]"], s=1)
plt.xlabel("Longitude [m]")
plt.ylabel("Latitude [m]")
plt.show()

In [ ]:
signal = data["Total field [nT]"]
fit = normal_fit(signal)
print(f"Normaltest: statistic={fit['statistic']:.4f}, p-value={fit['p_value']:.4e}")

x = np.linspace(signal.min(), signal.max(), 200)
plt.figure(figsize=(12, 6))
plt.hist(signal, bins=50, density=True, alpha=0.6, color="b", label="Signal Histogram")
plt.plot(x, fit["dist"].pdf(x), "r-", linewidth=2,
         label=f"Normal Fit\n$\mu$={fit['mu']:.2f}, $\sigma$={fit['sigma']:.2f}")
plt.xlabel("Signal Value", fontsize=16)
plt.ylabel("Probability Density", fontsize=16)
plt.title("Signal Histogram with Normal Distribution Fit", fontsize=18)
plt.legend(fontsize=14)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
filtered, Tx, Sx, ssq_freqs, scales = pr.propeller_ssq(data["Total field [nT]"], n_fft=512)

fmin, fmax = pr.PROPELLER_BAND
t = np.linspace(0, len(filtered) / FS, len(filtered))
band = (ssq_freqs >= fmin) & (ssq_freqs <= fmax)
edge = slice(pr.EDGE, -pr.EDGE)   # first and last second removed (edge artefacts)

fig, ax = plt.subplots(figsize=(20, 6))
pcm = ax.pcolormesh(t[edge], ssq_freqs[band], 20 * np.log10(np.abs(Tx[band, edge]) + 1e-12),
                    shading="auto", cmap="turbo")
cbar = fig.colorbar(pcm, ax=ax, shrink=0.9)
cbar.set_label("Magnitude (-)", fontsize=16)
cbar.ax.tick_params(labelsize=14)
ax.set_ylim(fmin, fmax)
ax.set_yticks(np.arange(fmin, fmax, 1))
ax.set_xlabel("Time (s)", fontsize=17)
ax.set_ylabel("Frequency (Hz)", fontsize=17)
ax.tick_params(labelsize=14)
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
ridge_idxs = pr.propeller_ridges(Tx, scales, n_ridges=4, bw=10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1, 2]})
axes[0].plot(filtered, color="b")
axes[0].set_title("x(t)", fontsize=16)
axes[0].set_xlabel("Time [samples]", fontsize=12)
axes[0].set_ylabel("Signal Amplitude [A.U.]", fontsize=12)
axes[0].grid(True, alpha=0.3)

im = axes[1].imshow(np.abs(Tx), aspect="auto", cmap="turbo", origin="lower")
axes[1].plot(ridge_idxs, linestyle="--", color="k")
axes[1].set_title("abs(SSQ_STFT) w/ ridge_idxs", fontsize=16)
axes[1].set_xlabel("Time [samples]", fontsize=12)
axes[1].set_ylabel("Frequency bin", fontsize=12)
axes[1].grid(True, alpha=0.3)
fig.colorbar(im, ax=axes[1], orientation="vertical", pad=0.02).set_label("Magnitude", fontsize=12)

plt.tight_layout()
plt.show()

## Ridge frequency and amplitude vs. velocity

Ridges of all velocity flights, only the samples within ±0.2 m/s of the flight's target velocity.
Ridge 1 is the lowest frequency.

In [ ]:
ridges = pr.velocity_ridges()

In [ ]:
sns.set_theme(style="ticks")
palette = sns.color_palette("pastel")

fig, axes = plt.subplots(5, 4, figsize=(15, 14), sharex=True, sharey=True)
fig.subplots_adjust(hspace=0.45, wspace=0.35)
for sensor in range(1, 6):
    for ridge in range(1, 5):
        ax = axes[sensor - 1, ridge - 1]
        df = ridges.get((sensor, ridge))
        if df is not None and len(df) > 1:
            x, y = df["Velocity [m/s]"].values, df["Frequencies"].values
            slope, intercept = pr.fit_line(x, y)
            x_fit = np.linspace(x.min(), x.max(), 100)

            ax.scatter(x, y, s=1, color=palette[sensor - 1], alpha=0.8)
            ax.plot(x_fit, slope * x_fit + intercept, color="black", linestyle="--", linewidth=1)
            ax.text(0.5, 1.05, f"S{sensor} | R{ridge} | slope = {slope:.2f}", transform=ax.transAxes,
                    ha="center", va="bottom", fontsize=12)

        ax.set_yticks(np.arange(36, 61, 3))
        ax.tick_params(top=True, right=True, direction="in", length=3, labelsize=12)
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.8)

fig.text(0.5, 0.04, "Velocity [m/s]", ha="center", fontsize=14)
fig.text(0.04, 0.5, "Frequency [Hz]", va="center", rotation="vertical", fontsize=14)
plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.96])
plt.show()

In [ ]:
sns.set_theme(style="whitegrid")
palette = sns.color_palette("pastel")

fig, axes = plt.subplots(5, 4, figsize=(15, 15), sharex=True, sharey=False)
fig.subplots_adjust(hspace=0.5, wspace=0.4)
for sensor in range(1, 6):
    for ridge in range(1, 5):
        ax = axes[sensor - 1, ridge - 1]
        df = ridges.get((sensor, ridge))
        if df is None or df.empty:
            ax.text(0.5, 0.5, "No Data", transform=ax.transAxes, ha="center", va="center", fontsize=9, color="gray")
        else:
            x, y = pr.amplitude_db(df)
            ax.scatter(x, y, s=10, color=palette[sensor - 1], alpha=0.8)
            title = f"S{sensor} | R{ridge}"
            if len(x) > 1:
                slope, intercept = pr.fit_line(x, y)
                x_fit = np.linspace(x.min(), x.max(), 100)
                ax.plot(x_fit, slope * x_fit + intercept, color="black", linestyle="--", linewidth=1)
                title += f" | slope={slope:.2f}"
            ax.set_title(title, fontsize=12, loc="left")

            y_pad = (y.max() - y.min()) * 0.1 if y.max() != y.min() else 1
            ax.set_ylim(y.min() - y_pad, y.max() + y_pad)
            tick_step = max((y.max() - y.min()) / 4, 1)
            ax.set_yticks(np.arange(y.min(), y.max() + tick_step, tick_step))

        ax.tick_params(top=True, right=True, direction="in", length=3, labelsize=12)
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.8)

fig.text(0.5, 0.04, "Velocity [m/s]", ha="center", fontsize=14)
fig.text(0.04, 0.5, "Power spectral density [dB]", va="center", rotation="vertical", fontsize=14)
plt.tight_layout(rect=[0.05, 0.05, 0.95, 0.97])
plt.show()